# Part B — RNN / LSTM / GRU (Fast Version)
### Time-Series Prediction on Airline Passengers Dataset
**Models:** RNN vs LSTM vs GRU  
**Metric:** RMSE (Root Mean Squared Error)  
**Run Time:** ~3-5 minutes total

In [ ]:
# CELL 1 — Install Libraries
import sys
!{sys.executable} -m pip install torch numpy matplotlib scikit-learn pandas --quiet
print('✅ Done!')

In [ ]:
# CELL 2 — Imports & Settings
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
import math, time, os

torch.manual_seed(42)
np.random.seed(42)

DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EPOCHS    = 50        # ✅ Fast — only 50 epochs
LR        = 0.01
SEQ_LEN   = 12        # use 12 months to predict next month
HIDDEN    = 32        # ✅ Small hidden size = faster
LAYERS    = 1         # ✅ Single layer = faster

os.makedirs('./outputs', exist_ok=True)
print(f'✅ Device: {DEVICE} | Epochs: {EPOCHS} | Hidden: {HIDDEN}')

In [ ]:
# CELL 3 — Load Airline Passengers Dataset (built-in, no download needed)
# Classic dataset: monthly airline passengers 1949-1960 (144 data points)
data_url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv'

try:
    df = pd.read_csv(data_url, header=0, usecols=[1])
    print('✅ Dataset loaded from internet!')
except:
    # Fallback: hardcoded dataset if no internet
    values = [
        112,118,132,129,121,135,148,148,136,119,104,118,
        115,126,141,135,125,149,170,170,158,133,114,140,
        145,150,178,163,172,178,199,199,184,162,146,166,
        171,180,193,181,183,218,230,242,209,191,172,194,
        196,196,236,235,229,243,264,272,237,211,180,201,
        204,188,235,227,234,264,302,293,259,229,203,229,
        242,233,267,269,270,315,364,347,312,274,237,278,
        284,277,317,313,318,374,413,405,355,306,271,306,
        315,301,356,348,355,422,465,467,404,347,305,336,
        340,318,362,348,363,435,491,505,404,359,310,337,
        360,342,406,396,420,472,548,559,463,407,362,405,
        417,391,419,461,472,535,622,606,508,461,390,432
    ]
    df = pd.DataFrame(values, columns=['Passengers'])
    print('✅ Dataset loaded from fallback (hardcoded)!')

# Normalize to [0, 1]
scaler  = MinMaxScaler()
dataset = scaler.fit_transform(df.values.astype(float))

print(f'✅ Total data points : {len(dataset)}')

# Plot raw data
plt.figure(figsize=(10, 3))
plt.plot(df.values, color='steelblue', linewidth=2)
plt.title('Airline Passengers Dataset (Raw)', fontweight='bold')
plt.xlabel('Month'); plt.ylabel('Passengers')
plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig('./outputs/raw_data.png', dpi=150)
plt.show()
print('✅ Saved: raw_data.png')

In [ ]:
# CELL 4 — Create Sequences & Train/Test Split
def make_sequences(data, seq_len):
    """Turn time series into (input_sequence → next_value) pairs."""
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i : i + seq_len])
        y.append(data[i + seq_len])
    return np.array(X), np.array(y)

X, y = make_sequences(dataset, SEQ_LEN)

# 80% train, 20% test
split    = int(len(X) * 0.8)
X_train  = torch.tensor(X[:split],  dtype=torch.float32).to(DEVICE)
y_train  = torch.tensor(y[:split],  dtype=torch.float32).to(DEVICE)
X_test   = torch.tensor(X[split:],  dtype=torch.float32).to(DEVICE)
y_test   = torch.tensor(y[split:],  dtype=torch.float32).to(DEVICE)

# Shape: (samples, seq_len, 1)
X_train = X_train.unsqueeze(-1)
X_test  = X_test.unsqueeze(-1)

print(f'✅ X_train: {X_train.shape} | X_test: {X_test.shape}')
print(f'   Train samples: {len(X_train)} | Test samples: {len(X_test)}')

In [ ]:
# CELL 5 — Define RNN, LSTM, GRU Models
class SequenceModel(nn.Module):
    """
    One class handles all 3 model types.
    Just change model_type to 'RNN', 'LSTM', or 'GRU'.
    """
    def __init__(self, model_type='LSTM', input_size=1, hidden=32, layers=1):
        super().__init__()
        self.model_type = model_type

        if model_type == 'RNN':
            self.rnn = nn.RNN(input_size, hidden, layers,
                              batch_first=True, dropout=0.0)
        elif model_type == 'LSTM':
            self.rnn = nn.LSTM(input_size, hidden, layers,
                               batch_first=True, dropout=0.0)
        elif model_type == 'GRU':
            self.rnn = nn.GRU(input_size, hidden, layers,
                              batch_first=True, dropout=0.0)

        self.fc = nn.Linear(hidden, 1)   # output: 1 value (next passenger count)

    def forward(self, x):
        out, _ = self.rnn(x)             # out: (batch, seq_len, hidden)
        out     = self.fc(out[:, -1, :]) # take last time step → (batch, 1)
        return out


# Create all 3 models
rnn_model  = SequenceModel('RNN',  hidden=HIDDEN, layers=LAYERS).to(DEVICE)
lstm_model = SequenceModel('LSTM', hidden=HIDDEN, layers=LAYERS).to(DEVICE)
gru_model  = SequenceModel('GRU',  hidden=HIDDEN, layers=LAYERS).to(DEVICE)

params = sum(p.numel() for p in rnn_model.parameters())
print(f'✅ All 3 models ready!')
print(f'   Parameters per model : ~{params:,}')
print(f'   RNN  : {rnn_model.rnn}')
print(f'   LSTM : {lstm_model.rnn}')
print(f'   GRU  : {gru_model.rnn}')

In [ ]:
# CELL 6 — Training Function
def train_model(model, name):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    losses    = []
    start     = time.time()

    print(f'\n🚀 Training {name}...')
    for epoch in range(1, EPOCHS + 1):
        model.train()
        optimizer.zero_grad()
        output = model(X_train)                    # forward
        loss   = criterion(output, y_train)        # MSE loss
        loss.backward()                            # backward
        optimizer.step()                           # update
        losses.append(loss.item())

        if epoch % 10 == 0:
            print(f'  Epoch {epoch:>3}/{EPOCHS} | Loss: {loss.item():.6f}')

    elapsed = time.time() - start
    print(f'⏱ Done in {elapsed:.1f} sec')
    return losses, elapsed

print('✅ Training function ready!')

In [ ]:
# CELL 7 — Train All 3 Models (~1 min total)
losses_rnn,  t_rnn  = train_model(rnn_model,  'RNN')
losses_lstm, t_lstm = train_model(lstm_model, 'LSTM')
losses_gru,  t_gru  = train_model(gru_model,  'GRU')

In [ ]:
# CELL 8 — Plot Loss Curves (All 3 on One Chart)
plt.figure(figsize=(10, 4))
plt.plot(losses_rnn,  label='RNN',  color='tomato',     linewidth=2)
plt.plot(losses_lstm, label='LSTM', color='steelblue',  linewidth=2)
plt.plot(losses_gru,  label='GRU',  color='seagreen',   linewidth=2)
plt.title('Training Loss Curves — RNN vs LSTM vs GRU', fontsize=13, fontweight='bold')
plt.xlabel('Epoch'); plt.ylabel('MSE Loss')
plt.legend(fontsize=11); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('./outputs/loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: loss_curves.png')

In [ ]:
# CELL 9 — Predictions & RMSE
def get_rmse(model, name):
    model.eval()
    with torch.no_grad():
        preds = model(X_test).cpu().numpy()
    actual = y_test.cpu().numpy()

    # Inverse transform back to original scale
    preds_inv  = scaler.inverse_transform(preds)
    actual_inv = scaler.inverse_transform(actual)

    rmse = math.sqrt(mean_squared_error(actual_inv, preds_inv))
    print(f'  {name:<6} RMSE: {rmse:.2f}')
    return preds_inv, actual_inv, rmse

print('📊 Test RMSE (lower = better):')
preds_rnn,  actual, rmse_rnn  = get_rmse(rnn_model,  'RNN')
preds_lstm, actual, rmse_lstm = get_rmse(lstm_model, 'LSTM')
preds_gru,  actual, rmse_gru  = get_rmse(gru_model,  'GRU')

In [ ]:
# CELL 10 — Plot Predictions vs Actual
x_axis = range(len(actual))

fig, axes = plt.subplots(3, 1, figsize=(12, 10))
fig.suptitle('Actual vs Predicted — Airline Passengers', fontsize=14, fontweight='bold')

for ax, preds, name, color, rmse in [
    (axes[0], preds_rnn,  'RNN',  'tomato',    rmse_rnn),
    (axes[1], preds_lstm, 'LSTM', 'steelblue', rmse_lstm),
    (axes[2], preds_gru,  'GRU',  'seagreen',  rmse_gru),
]:
    ax.plot(x_axis, actual,        label='Actual',    color='black',  linewidth=2)
    ax.plot(x_axis, preds.flatten(),label=f'{name} Predicted', color=color, linewidth=2, linestyle='--')
    ax.set_title(f'{name}  |  RMSE: {rmse:.2f}', fontweight='bold')
    ax.set_xlabel('Time Step'); ax.set_ylabel('Passengers')
    ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('./outputs/predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: predictions.png')

In [ ]:
# CELL 11 — RMSE Bar Chart
names  = ['RNN', 'LSTM', 'GRU']
rmses  = [rmse_rnn, rmse_lstm, rmse_gru]
colors = ['tomato', 'steelblue', 'seagreen']

plt.figure(figsize=(7, 4))
bars = plt.bar(names, rmses, color=colors, edgecolor='black', width=0.4)
for bar, val in zip(bars, rmses):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.5,
             f'{val:.2f}', ha='center', fontweight='bold', fontsize=12)
plt.title('RMSE Comparison — RNN vs LSTM vs GRU\n(Lower is Better)',
          fontsize=13, fontweight='bold')
plt.ylabel('RMSE'); plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('./outputs/rmse_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: rmse_comparison.png')

In [ ]:
# CELL 12 — Final Summary Table
params_rnn  = sum(p.numel() for p in rnn_model.parameters())
params_lstm = sum(p.numel() for p in lstm_model.parameters())
params_gru  = sum(p.numel() for p in gru_model.parameters())

print('='*58)
print('          FINAL COMPARISON — RNN vs LSTM vs GRU')
print('='*58)
print(f'{"Metric":<22} {"RNN":>10} {"LSTM":>12} {"GRU":>10}')
print('-'*58)
print(f'{"Test RMSE":<22} {rmse_rnn:>10.2f} {rmse_lstm:>12.2f} {rmse_gru:>10.2f}')
print(f'{"Parameters":<22} {params_rnn:>10,} {params_lstm:>12,} {params_gru:>10,}')
print(f'{"Training Time (sec)":<22} {t_rnn:>10.1f} {t_lstm:>12.1f} {t_gru:>10.1f}')
print(f'{"Final Loss":<22} {losses_rnn[-1]:>10.6f} {losses_lstm[-1]:>12.6f} {losses_gru[-1]:>10.6f}')
print('='*58)

best = names[rmses.index(min(rmses))]
print(f'\n🏆 Best Model: {best} (lowest RMSE = {min(rmses):.2f})')
print('\n📁 Outputs saved in ./outputs/')
print('   → raw_data.png')
print('   → loss_curves.png')
print('   → predictions.png')
print('   → rmse_comparison.png')
print('\n🎉 Part B Complete! Move to Part C (GAN).')